In [1]:
import numpy as np
import pandas as pd

## Load Data

In [2]:
data = pd.read_csv('mnist.csv')
data.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
data.describe()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
count,70000.000000,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,...,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.0,70000.0,70000.0,70000.0
mean,4.452429,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.197414,0.099543,0.046629,0.016614,0.012957,0.001714,0.0,0.0,0.0,0.0
std,2.890195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.991206,4.256304,2.783732,1.561822,1.553796,0.320889,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,254.000000,254.000000,253.000000,253.000000,254.000000,62.000000,0.0,0.0,0.0,0.0


In [4]:
data['label'].unique()

array([5, 0, 4, 1, 9, 2, 3, 6, 7, 8])

In [5]:
X = data.iloc[:,1:]
y = data.iloc[:,0]

## Standardization and Splitting the data

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_train,X_test,y_train,y_test = train_test_split(X.to_numpy(),y.to_numpy(),test_size=0.2,random_state=1234)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [7]:
print(f'size of X_train = {X_train.shape}')
print(f'size of X_test = {X_test.shape}')
print(f'size of y_train = {y_train.shape}')
print(f'size of y_test = {y_test.shape}')

size of X_train = (56000, 784)
size of X_test = (14000, 784)
size of y_train = (56000,)
size of y_test = (14000,)


## Applying one hot encoding

In [8]:
y_train = pd.get_dummies(pd.DataFrame(y_train)[0])
y_train

,0,1,2,3,4,5,6,7,8,9
0,0,0,0,0,0,1,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0
2,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
55995,0,0,1,0,0,0,0,0,0,0
55996,1,0,0,0,0,0,0,0,0,0
55997,1,0,0,0,0,0,0,0,0,0
55998,1,0,0,0,0,0,0,0,0,0


In [9]:
y_train = y_train.to_numpy()
y_train.shape

(56000, 10)

## Model implementation

In [10]:
class NN:
    def __init__(self,alpha=0.01,epochs=2500,threshold=0.5):
        self.alpha = alpha
        self.epochs = epochs
        self.threshold = threshold
        self.weights = []
        self.biases = []
#         self.activatios = []
        self.clas = None
    def forward_prop(self,X,weights,biases):
        a = X.T
        activations = [a]
        for w,b in zip(weights,biases):
            z = w @ a + b
            a = self.sigmoid(z)
            activations.append(a)
        return activations

    def backward_prop(self,X,y,weights,biases,num_of_layers):
        activations = self.forward_prop(X,weights,biases)
        w_updates = [np.zeros(w.shape) for w in weights]
        b_updates = [np.zeros(b.shape) for b in biases]
        
        delta_z = activations[-1] - y
        w_updates[-1] = delta_z @ activations[-2].T
        b_updates[-1] = np.sum(delta_z,axis=1,keepdims=True)
        
        for l in range(2,num_of_layers):
            delta_z = (weights[-l+1].T @ delta_z) * (activations[-l]*(1-activations[-l]))
            w_updates[-l] = delta_z @ activations[-l-1].T
            b_updates[-l] = np.sum(delta_z,axis=1,keepdims=True)
            
        return w_updates,b_updates
        
    def fit(self,X,y,num_of_layers, size_of_layers):
        
        m,features = X.shape
        self.clas = y.shape[1]
        for _  in range(self.clas):
            weights = []
            biases = []
            np.random.seed(20)
            w = np.random.rand(size_of_layers[0],features)
            b = np.zeros((size_of_layers[0],1))
            weights.append(w)
            biases.append(b)

            for i in range(1,num_of_layers):
                # w[layer] = (#nodes in the layer ,#nodes in previous layer) 
                np.random.seed(25)
                w = np.random.rand(size_of_layers[i],size_of_layers[i-1])
                b = np.zeros((size_of_layers[i],1))
                weights.append(w)
                biases.append(b)
            self.weights.append(weights)
            self.biases.append(biases)

        for i in range(self.epochs):
            for j in range(self.clas):
                w_updates,b_updates = self.backward_prop(X,y[:,j],self.weights[j],self.biases[j],num_of_layers)
                
                self.weights[j] = [w - self.alpha * dw for w, dw in zip(self.weights[j], w_updates)]
                self.biases[j] = [b - self.alpha * db for b, db in zip(self.biases[j], b_updates)]
                
    def sigmoid(self,z):
        z = np.float128(z)
        return 1/(1+np.exp(-z))
    def soft_max(self,z):
        numerator  = np.exp(z).T
        denominator = np.sum(np.exp(z),axis=1)
        soft = numerator/denominator
        return soft.T
    def predict(self,X):
        m = X.shape[0]
        phiz = np.zeros((m,self.clas))
        for j in range(self.clas):
            activation = self.forward_prop(X,self.weights[j],self.biases[j])
            phiz[:,j] = activation[-1] 
        f_wb = self.soft_max(phiz)
        y_pred = np.argmax(f_wb,axis=1)
        return y_pred
    
    def cost(self,y_pred,y):
        return np.mean((y_pred-y)**2)
    def accuracy(self,y_pred,y):
        return np.mean(y_pred==y)

In [10]:
# 2 layers 6 , 1

In [15]:
model1 = NN()
model1.fit(X_train,y_train,2,[6,1])

In [33]:
y_pred = model1.predict(X_test)

In [34]:
print(f"Accuracy = { model1.accuracy(y_pred,y_test) *100 :0.2f}%" )

Accuracy = 14.61%


In [21]:
# 3 layers 6 , 4 , 1

In [22]:
model2 = NN()
model2.fit(X_train,y_train,3,[6,4,1])

In [35]:
y_pred = model2.predict(X_test)

In [36]:
print(f"Accuracy = { model2.accuracy(y_pred,y_test) *100 :0.2f}%" )

Accuracy = 10.34%


In [26]:
# 3 layers 4 , 6 , 1

In [27]:
model3 = NN()
model3.fit(X_train,y_train,3,[4,6,1])

In [37]:
y_pred = model3.predict(X_test)

In [38]:
print(f"Accuracy = { model3.accuracy(y_pred,y_test) *100 :0.2f}%" )

Accuracy = 9.55%


In [31]:
# Accuracies for all models :

# model 1 - 2 layers 6 , 1 : Accuracy = 14.61%
# ___________________________________________________
# model 2 - 3 layers 6 , 4 , 1 : Accuracy = 10.34%
# ___________________________________________________
# model 3 - 3 layers 4 , 6 , 1 : Accuracy = 9.55%

